# M05 actual-MACE TPU component probe

First reproduce CPU science and export the CPU reference. Then select **Runtime → Change runtime type → TPU**. Runtime changes can erase `/content`; restore the complete verified reference and exact source. This experiment does not move OpenMM/PME/integration onto TPU. Unsupported precision/operators and missing hardware remain explicit outcomes. Never change weights or silently lower precision.


In [ ]:
from pathlib import Path
# Paste the exact reviewed implementation commit printed in notebooks/README.md.
SOURCE_SHA = ""
REFERENCE = Path("/content/restored-cpu-reference/reference.json")
REFERENCE_SHA256 = ""  # identity.json from the immutable, verified CPU export
REPO = Path("/content/AToM_MLMM")
CPU_ROOT = Path("/content/atom-mlmm-colab-cpu")
TPU_PREFIX = Path("/content/atom-mlmm-colab-tpu")
EVIDENCE = Path("/content/m05-tpu-attempt-01")  # unused; preserve prior attempts
MODEL_SHA256 = "165cce4cfec5a34b9c64d4ebf95de15d71106bb584b7291c8470f0749977c46f"
assert len(SOURCE_SHA)==40 and all(c in "0123456789abcdef" for c in SOURCE_SHA)
assert not EVIDENCE.exists(), "Choose an unused attempt; prior evidence is retained."
SETUP = CPU_ROOT


In [ ]:
import datetime, json, os, signal, subprocess, time

def run_command(argv, label, timeout, cwd=None, env=None):
    """Bound a subprocess; retain full logs and exit status, including failures."""
    EVIDENCE.mkdir(parents=True, exist_ok=True)
    log = EVIDENCE / (label + ".log")
    status = EVIDENCE / (label + "-command.json")
    assert not log.exists() and not status.exists(), "Command label already used."
    started = time.monotonic()
    record = {"argv":list(map(str,argv)), "cwd":str(cwd or REPO),
              "utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
              "timeout_seconds":timeout, "timed_out":False, "exit_code":None}
    with log.open("w") as output:
        process = subprocess.Popen(argv, cwd=cwd or REPO, env=env,
                                   stdout=output, stderr=subprocess.STDOUT,
                                   start_new_session=True)
        try:
            record["exit_code"] = process.wait(timeout=timeout)
        except subprocess.TimeoutExpired:
            record["timed_out"] = True
            os.killpg(process.pid, signal.SIGKILL)
            record["exit_code"] = process.wait()
            raise
        finally:
            record["elapsed_seconds"] = time.monotonic()-started
            status.write_text(json.dumps(record,indent=2)+"\n")
    print(label, "exit", record["exit_code"], "log:",log)
    if record["exit_code"]:
        raise subprocess.CalledProcessError(record["exit_code"],argv)

# Every computation is a fresh Bash subprocess with explicit main activation.
import shlex
def cpu_command(command, label, timeout=900):
    prefix = "set -euo pipefail; source " + shlex.quote(str(SETUP/"activate.sh"))
    prefix += '; export OPENBLAS_NUM_THREADS=2 OMP_NUM_THREADS=2 PYTHONPATH="$PWD/src"; '
    return run_command(["bash","-c",prefix+command],label,timeout)


In [ ]:
import hashlib, platform, shutil
assert platform.system()=="Linux" and platform.machine()=="x86_64"
if not (REPO/".git").exists():
    run_command(["git","clone","--no-checkout","https://github.com/georgpou/AToM_MLMM.git",str(REPO)],"clone",600,cwd=Path("/content"))
run_command(["git","fetch","origin",SOURCE_SHA],"fetch-exact-source",600)
run_command(["git","checkout","--detach",SOURCE_SHA],"checkout-exact-source",60)
actual=subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip()
assert actual==SOURCE_SHA
assert hashlib.sha256((REPO/"models/mace-off23-small/MACE-OFF23_small.model").read_bytes()).hexdigest()==MODEL_SHA256
record={"utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),"source_sha":actual,
        "model_sha256":MODEL_SHA256,"platform":platform.platform(),"cpu_count":os.cpu_count(),
        "memory_kib":[l.strip() for l in Path("/proc/meminfo").read_text().splitlines() if l.startswith(("MemTotal:","MemAvailable:"))],
        "free_disk_bytes":shutil.disk_usage("/content").free,"threads":2,"real_atoms":{"abfe":224,"rbfe":233},
        "workers":0,"seeds":"fixed-coordinate probe; no MD","steps":0,"wall_limit_s":900,
        "persistence":"each completed comparison in report.json; all negative diagnostics retained"}
(EVIDENCE/"resources.json").write_text(json.dumps(record,indent=2)+"\n")


## CPU reference before switching runtimes

Run these cells on the CPU runtime with the unchanged locks. They load the same approved checkpoint and save ABFE/RBFE both-map energy, coordinate gradients, all-real cap projection, changed-coordinate and finite-difference inputs. Save the **complete stopped attempt** with its hash seal before switching runtimes. Copy the reference SHA from identity.json; it must be supplied on restoration. Skip reference generation only when restoring that verified export.


In [ ]:
run_command(["bash","environment/cloud-cpu/install.sh"],"install-unchanged-cpu",1800,
            env={**os.environ,"ATOM_MLMM_SETUP_ROOT":str(CPU_ROOT)})
cpu_command("python environment/cloud-cpu/validate.py --root "+str(CPU_ROOT)+" --repository "+str(REPO)+" --strict", "strict-cpu-profile", 900)


In [ ]:
cpu_command("python tools/m05_tpu_probe.py cpu-reference --output "+str(EVIDENCE/"cpu-reference"),"actual-cpu-reference",900)
print((EVIDENCE/"cpu-reference/identity.json").read_text())


## TPU runtime: restore and verify first

After selecting TPU, rerun the parameter/helper/source cells with a new attempt directory and the same exact source. Install the unchanged CPU base in this fresh runtime, restore the complete verified CPU export, then set REFERENCE and REFERENCE_SHA256. Do not claim a hardware pass from installation or selecting TPU. A nonzero probe exit preserves report.json, exception.txt and metrics; export those as a negative result.


In [ ]:
assert len(REFERENCE_SHA256)==64
assert hashlib.sha256(REFERENCE.read_bytes()).hexdigest()==REFERENCE_SHA256
run_command(["bash","environment/colab-tpu/install.sh",str(CPU_ROOT),str(TPU_PREFIX)],"install-separate-xla-profile",1800)
run_command([str(TPU_PREFIX/"bin/python"),"-m","pip","freeze"],"experimental-inventory",60)


In [ ]:
tpu_env={**os.environ,"PJRT_DEVICE":"TPU","PYTHONPATH":str(REPO/"src"),
         "OMP_NUM_THREADS":"2","MKL_NUM_THREADS":"2","OPENBLAS_NUM_THREADS":"2"}
for key in ("XLA_USE_BF16","XLA_DOWNCAST_BF16","TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"):
    tpu_env.pop(key,None)
run_command([str(TPU_PREFIX/"bin/python"),"tools/m05_tpu_probe.py","tpu",str(REFERENCE),
             "--reference-sha256",REFERENCE_SHA256,"--output",str(EVIDENCE/"actual-tpu")],
            "actual-mace-tpu",900,env=tpu_env)
print((EVIDENCE/"actual-tpu/report.json").read_text()[:1500])


## Export complete stopped attempts

Run the export cell after a pass **or a retained negative result**. Active transactions stay on local `/content`. Set an unused destination on storage you authorize; authenticate it yourself. Returned evidence must pass inert hash verification before any Colab/TPU claim. No secrets or large scientific output belongs in notebook cells.


In [ ]:
DURABLE_DESTINATION = ""  # unused directory on authorized mounted storage
assert DURABLE_DESTINATION, "Set an unused storage destination."
# All jobs have stopped; this export creates no log inside the source tree.
payload="from atm_mlmm.evidence import export_evidence; import sys; print(export_evidence(sys.argv[1],sys.argv[2]))"
command="set -euo pipefail; source "+shlex.quote(str(SETUP/"activate.sh"))
command += '; export PYTHONPATH="$PWD/src"; python -c '+shlex.quote(payload)
command += " "+shlex.quote(str(EVIDENCE))+" "+shlex.quote(DURABLE_DESTINATION)
subprocess.run(["bash","-c",command],cwd=REPO,check=True)
